## Wiki Scrape & Export (Files Only)

Scrape the Gitit-powered RC wiki (`rci.hpc.virginia.edu/wiki`) and save each page as a local `.md` file.

This trims `scrape-wiki-new.ipynb` down to only the fetch + export steps, dropping the DB/vector-store ingestion — matching the pattern used in `scrape-jira-new.ipynb` and `scrape-md-only.ipynb`.

## 1. NetBadge Login: Generate `auth_cookies.json`

Set `WIKI_USERNAME` (your UVA computing ID) and `WIKI_PASSWORD` in the
repository-root `.env`, using `.env.example` as a template. Existing environment
variables take precedence. `.env.example` contains placeholders, not credentials.

One-time browser setup: `playwright install chromium`.

This cell reads credentials without prompting, so it works with nbconvert.
You still need to approve the Duo push on your phone. Run it again when cookies
expire. Cookies are saved to `scrapers/auth_cookies.json`.


In [ ]:
import json as _json
import os
import time
from pathlib import Path

from dotenv import load_dotenv

# Resolve configuration consistently for nbconvert and interactive notebooks.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app" / "kb_integration" / "tasks.py").is_file()
     and (path / "scrapers").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or a subfolder.")

load_dotenv(PROJECT_ROOT / ".env", override=False)
_username = os.getenv("WIKI_USERNAME", "").strip()
_password = os.getenv("WIKI_PASSWORD", "")
_missing = [name for name, value in (
    ("WIKI_USERNAME", _username), ("WIKI_PASSWORD", _password)
) if not value.strip()]
if _missing:
    raise RuntimeError(
        "Missing wiki credentials: " + ", ".join(_missing)
        + ". Set them in the repository .env using .env.example as a template "
        "or provide environment variables."
    )

from playwright.async_api import async_playwright

LOGIN_TARGET = "https://rci.hpc.virginia.edu/wiki/Special:AllPages"
AUTH_COOKIES_PATH = PROJECT_ROOT / "scrapers" / "auth_cookies.json"

USERNAME_SEL = "input[name='j_username'], input[name='username'], input[id='username']"
PASSWORD_SEL = "input[name='j_password'], input[name='password'], input[id='password']"


async def _fill_credentials(page, username, password, label=""):
    """Fill username/password and submit by pressing Enter on the password field."""
    await page.locator(USERNAME_SEL).first.fill(username)
    await page.locator(PASSWORD_SEL).first.fill(password)
    # Press Enter on the password field — submits the password form only,
    # not the certificate-login button which is a separate form/button.
    await page.locator(PASSWORD_SEL).first.press("Enter")
    await page.wait_for_load_state("domcontentloaded", timeout=30000)
    await page.wait_for_timeout(2000)
    print(f"  {label}URL after submit : {page.url}")


async def netbadge_login(username, password, target=LOGIN_TARGET, output=AUTH_COOKIES_PATH):
    """Log in via NetBadge + Duo in a headless browser and save cookies to `output`.

    Uses Playwright's async API because the sync API cannot run inside the
    asyncio event loop that Jupyter/IPython kernels already have running.
    """

    print("Launching headless browser ...", flush=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context()
        page = await ctx.new_page()

        await page.goto(target, timeout=30000)
        await page.wait_for_load_state("domcontentloaded", timeout=20000)
        await page.wait_for_timeout(2000)

        # ── Step 1: handle certificate-auth failure page ───────────────
        for _ in range(4):
            if await page.locator("#mw-content-text").count() > 0:
                break  # already on the wiki

            if "my.policy" in page.url or "Certificate Login Failed" in await page.content():
                print("  Certificate auth failed page — clicking Continue ...")
                await page.locator(
                    "a:has-text('Continue'), input[value='Continue'], button:has-text('Continue')"
                ).first.click()
                await page.wait_for_load_state("domcontentloaded", timeout=20000)
                await page.wait_for_timeout(2000)
                print(f"  Now at: {page.url}")

            if await page.locator(USERNAME_SEL).count() > 0:
                await _fill_credentials(page, username, password, label="[login] ")
                break

        # ── Step 2: handle Duo 2FA ──────────────────────────────────────
        if await page.locator("#mw-content-text").count() == 0:
            await page.wait_for_timeout(3000)
            print(f"  Pre-Duo URL   : {page.url}")
            print(f"  Pre-Duo title : {await page.title()}")

            # Some Duo setups require clicking "Send Me a Push" instead of
            # auto-sending. Try common button labels before waiting.
            for push_text in ["Send Me a Push", "Send push", "Push", "Send request"]:
                push_btn = page.locator(
                    f"button:has-text('{push_text}'), input[value='{push_text}'], a:has-text('{push_text}')"
                )
                if await push_btn.count() > 0:
                    print(f"  Clicking Duo button: '{push_text}'")
                    await push_btn.first.click()
                    await page.wait_for_timeout(1500)
                    break

            # Dump what's actually on the Duo page — useful if the push
            # still doesn't fire, since headless browsers can be blocked
            # by Duo's bot detection or need a different button/device pick.
            try:
                body_text = await page.locator("body").inner_text()
                print(f"  Duo page text (first 500 chars):\n{body_text[:500]}\n")
                await page.screenshot(path="debug_duo.png")
                print("  Screenshot saved -> scrapers/debug_duo.png")
            except Exception as e:
                print(f"  [warn] Could not capture Duo page debug info: {e}")

            print()
            print("Waiting for you to approve the Duo push on your phone (up to 90 seconds) ...", flush=True)

        # ── Step 3: wait for wiki — print status every 5 s ──────────────
        print("Waiting for wiki to load ...", flush=True)
        deadline = time.time() + 120
        while time.time() < deadline:
            cur_url = page.url
            if "rci.hpc.virginia.edu/wiki/" in cur_url and \
               "shibidp" not in cur_url and "duosecurity" not in cur_url:
                break
            print(f"  [{int(deadline - time.time())}s left] URL: {page.url} | title: {await page.title()}", flush=True)
            for btn_text in ["Trust this browser", "Stay signed in", "Yes", "Continue", "Proceed"]:
                btn = page.locator(f"button:has-text('{btn_text}'), input[value='{btn_text}']")
                if await btn.count() > 0:
                    print(f"  Clicking intermediate button: '{btn_text}'")
                    await btn.first.click()
                    await page.wait_for_load_state("domcontentloaded", timeout=15000)
                    break
            await page.wait_for_timeout(5000)
        else:
            print(f"[ERROR] Timed out. Final URL: {page.url}")
            try:
                await page.screenshot(path="debug_final.png")
                print("  Screenshot saved -> scrapers/debug_final.png")
            except Exception as e:
                print(f"  [warn] Could not capture final debug screenshot: {e}")
            await browser.close()
            raise RuntimeError("NetBadge/Duo login timed out.")

        print("Login successful!")

        cookies = await ctx.cookies()
        with open(output, "w") as f:
            _json.dump(cookies, f, indent=2)

        print(f"Saved {len(cookies)} cookies → {output}")
        await browser.close()


# No stdin prompts: compatible with nbconvert. Duo approval is still required.
print("Starting NetBadge login using configured wiki credentials ...", flush=True)
try:
    await netbadge_login(_username, _password)
finally:
    del _username, _password


## 2. Setup: Session & Helper Functions

Load the cookies generated by Step 1 in `scrapers/auth_cookies.json`.

**Wiki engine:** Gitit (not MediaWiki).  
- All pages: `_index`  
- Content selector: `#wikipage`  
- TOC: `#TOC`  
- Internal pages: `/wiki/Page%20Name`  
- Special URLs to skip: `_edit/`, `_history/`, `_discuss/`, `_delete/`, `_login`, `_logout`

In [2]:
import json
import os
import re
import time
from urllib.parse import urljoin, urlparse, unquote

import requests
from bs4 import BeautifulSoup

BASE_URL      = "https://rci.hpc.virginia.edu/wiki/"
INDEX_URL     = "https://rci.hpc.virginia.edu/wiki/_index"
COOKIES_FILE  = AUTH_COOKIES_PATH

SKIP_EXTENSIONS = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".svg",
                   ".pdf", ".zip", ".tar", ".gz", ".mp4", ".webp")

# Gitit special-path prefixes and excluded wiki sections
SKIP_PATH_SEGMENTS = (
    "/_edit/", "/_history/", "/_discuss/", "/_delete/",
    "/_login", "/_logout", "/_index", "/_categories", "/_random",
    "/_search", "/_upload", "/_activity", "/_export",
    "/OldWiki", "/OldOldWiki",
)

REQUEST_DELAY = 0.3


def load_cookies(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run the NetBadge login cell first.")
    with open(path) as f:
        raw = json.load(f)
    return raw


def make_session(cookies):
    sess = requests.Session()
    for c in cookies:
        sess.cookies.set(
            c["name"], c["value"],
            domain=c.get("domain", "").lstrip("."),
            path=c.get("path", "/"),
        )
    sess.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; UVA-RC-RAG-Scraper/1.0)",
    })
    return sess


def check_session(r):
    if "NetBadge" in r.text or "shibidp" in r.url:
        raise RuntimeError("NetBadge redirect — re-run the NetBadge login cell to refresh cookies.")


def should_skip_url(url):
    path = urlparse(url).path
    if path.lower().endswith(SKIP_EXTENSIONS):
        return True
    return any(seg in path for seg in SKIP_PATH_SEGMENTS)


def page_title_from_url(url):
    path = urlparse(url).path
    if path.startswith("/wiki/"):
        return unquote(path[len("/wiki/"):])
    return url


CLEAN_RE = [
    (re.compile(r"https?://\S+?\.(?:png|jpg|jpeg|gif|svg|webp)"), ""),
    (re.compile(r"\S+\.(?:png|jpg|jpeg|gif|svg|webp)"), ""),
    (re.compile(r"\n{3,}"), "\n\n"),
    (re.compile(r"[ \t]+"), " "),
]

def clean_text(text):
    for pattern, repl in CLEAN_RE:
        text = pattern.sub(repl, text)
    return text.strip()


# ── Load session ──────────────────────────────────────────────────────────────
raw_cookies = load_cookies(COOKIES_FILE)
sess = make_session(raw_cookies)
print(f"Loaded {len(raw_cookies)} cookies")

r = sess.get(BASE_URL, timeout=20)
try:
    check_session(r)
    print(f"Session OK — wiki reachable (HTTP {r.status_code})")
except RuntimeError as e:
    print(f"[ERROR] {e}")

Loaded 11 cookies
Session OK — wiki reachable (HTTP 200)


## 3. Enumerate Pages from `_index`

Fetch the full page list from `/wiki/_index`, then scrape each page's `#wikipage` content.  
Skips OldWiki / OldOldWiki pages and image files.

In [3]:
def get_all_page_urls():
    """Get all content page URLs from /wiki/_index."""
    r = sess.get(INDEX_URL, timeout=20)
    check_session(r)
    soup = BeautifulSoup(r.text, "html.parser")
    content = soup.find(id="wikipage") or soup.find(id="content")
    if not content:
        print("[ERROR] Could not find page listing on _index")
        return []

    urls = []
    for a in content.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        if text == "(delete)":
            continue
        full_url = urljoin(BASE_URL, href).split("#")[0]
        if not should_skip_url(full_url) and "/wiki/" in full_url:
            urls.append(full_url)
    return list(dict.fromkeys(urls))  # deduplicate, preserve order


def extract_page(url):
    """Fetch a Gitit wiki page and extract text from #wikipage."""
    r = sess.get(url, timeout=20)
    check_session(r)
    if r.status_code != 200 or "text/html" not in r.headers.get("Content-Type", ""):
        return None

    soup = BeautifulSoup(r.text, "html.parser")
    wp = soup.find(id="wikipage")
    if not wp:
        return None

    # Remove TOC and edit links
    for el in wp.select("#TOC, .editsection, .footnotes"):
        el.decompose()
    for img in wp.find_all("img"):
        img.decompose()

    # Rewrite internal links to absolute markdown-style
    for a in wp.find_all("a", href=True):
        href = a["href"]
        if not href.startswith("http"):
            href = urljoin(url, href)
        a["href"] = href
        text = a.get_text(strip=True)
        if text:
            a.string = f"[{text}]({href})"

    text = clean_text(wp.get_text(separator="\n"))
    if not text:
        return None

    # Title from <h1> or page URL
    title_tag = wp.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else page_title_from_url(url)

    # Categories (Gitit puts them on the page if any)
    cat_div = soup.find(id="categories") or soup.find(class_="categories")
    categories = []
    if cat_div:
        categories = [a.get_text(strip=True) for a in cat_div.find_all("a")]

    return {
        "text": f"# {title}\n\n{text}" if not text.startswith("#") else text,
        "metadata": {
            "source_type": "wiki",
            "source": url,
            "chunk_number": 0,
            "tags": ["wiki"] + categories,
            "date_updated": None,
            "title": title,
        },
    }


# ── Run ───────────────────────────────────────────────────────────────────────
print("Collecting page URLs from _index ...")
page_urls = get_all_page_urls()
print(f"Found {len(page_urls)} pages (OldWiki/OldOldWiki excluded)\n")

documents = {}
for i, url in enumerate(page_urls, 1):
    print(f"  [{i}/{len(page_urls)}] {page_title_from_url(url)}")
    try:
        doc = extract_page(url)
        if doc:
            documents[url] = doc
        else:
            print(f"    (skipped — no content)")
    except RuntimeError:
        raise
    except Exception as e:
        print(f"    [error] {e}")
    time.sleep(REQUEST_DELAY)

print(f"\nPhase 1 done — {len(documents)} documents extracted")

Found 100 pages (OldWiki/OldOldWiki excluded)

  [1/100] 
  [2/100] 20230123 - Weekly Leadership Meeting Agenda
    (skipped — no content)
  [3/100] ACCORD Infrastructure
  [4/100] ACCORD
  [5/100] Accounting
  [6/100] Adding S3 Connectors aka Storage-Gateways and Collections
  [7/100] Adding nodes to GPFS cluster
  [8/100] Administration
  [9/100] Agendas 2023
  [10/100] BIOS
  [11/100] Configless Slurm
  [12/100] Creating Shared Collections
  [13/100] Data Sensitivity
  [14/100] Database Planning and Policies
  [15/100] Deploy RESTful API
  [16/100] Deploying New Nodes
  [17/100] EasyBuild
  [18/100] Education & Workshops
  [19/100] Ethernet
  [20/100] File-Based Storage Inventory
  [21/100] Front Page
  [22/100] GPFS Project Storage
  [23/100] Gitit User’s Guide
  [24/100] Globus Collection Creation
  [25/100] Globus Plus Provisioning
  [26/100] Globus
  [27/100] Guides
  [28/100] Guides/
    (skipped — no content)
  [29/100] HPC Accounting Workflow
  [30/100] HR/
    (skipped — no 

## 4. Phase 2: Link-Following Gap Fill

Follow internal links starting from the homepage to catch pages not listed in `_index`.

In [4]:
def crawl_links(start_url, documents):
    visited = set(documents.keys())
    queue = [start_url.split("#")[0]]
    netloc = urlparse(start_url).netloc

    while queue:
        url = queue.pop(0)
        if url in visited or should_skip_url(url):
            continue
        visited.add(url)

        try:
            r = sess.get(url, timeout=20)
            if r.status_code != 200:
                continue
            check_session(r)
            if "text/html" not in r.headers.get("Content-Type", ""):
                continue

            soup = BeautifulSoup(r.text, "html.parser")

            if url not in documents:
                wp = soup.find(id="wikipage")
                if wp:
                    for el in wp.select("#TOC, .editsection, .footnotes"):
                        el.decompose()
                    for img in wp.find_all("img"):
                        img.decompose()
                    for a in wp.find_all("a", href=True):
                        href = a["href"]
                        if not href.startswith("http"):
                            href = urljoin(url, href)
                        a["href"] = href
                        text = a.get_text(strip=True)
                        if text:
                            a.string = f"[{text}]({href})"

                    text = clean_text(wp.get_text(separator="\n"))
                    title_tag = wp.find("h1")
                    title = title_tag.get_text(strip=True) if title_tag else page_title_from_url(url)

                    if text:
                        documents[url] = {
                            "text": f"# {title}\n\n{text}" if not text.startswith("#") else text,
                            "metadata": {
                                "source_type": "wiki",
                                "source": url,
                                "chunk_number": 0,
                                "tags": ["wiki"],
                                "date_updated": None,
                                "title": title,
                            },
                        }
                        print(f"  [crawl] {page_title_from_url(url)}")

            # Enqueue new wiki links
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if href.startswith(("http://", "https://")):
                    if urlparse(href).netloc != netloc:
                        continue
                    next_url = href.split("#")[0]
                else:
                    next_url = urljoin(url, href).split("#")[0]

                if (
                    next_url not in visited
                    and not should_skip_url(next_url)
                    and urlparse(next_url).netloc == netloc
                    and "/wiki/" in next_url
                ):
                    queue.append(next_url)

            time.sleep(REQUEST_DELAY)

        except RuntimeError:
            raise
        except Exception as e:
            print(f"  [error] {url}: {e}")


before = len(documents)
crawl_links(BASE_URL, documents)
print(f"Phase 2 done — added {len(documents) - before} additional pages")
print(f"Total documents: {len(documents)}")

Phase 2 done — added 0 additional pages
Total documents: 92


## 5. Preview Scraped Content

In [5]:
print(f"Total wiki documents scraped: {len(documents)}")
print()

for url, d in list(documents.items())[:10]:
    print(f"  - {url}")
    print(f"    title : {d['metadata'].get('title', '(none)')}")
    print(f"    tags  : {d['metadata']['tags']}")
    print(f"    text  : {d['text'][:120].strip()}...")
    print()

Total wiki documents scraped: 92

  - https://rci.hpc.virginia.edu/wiki/
    title : Welcome to Research Computing!
    tags  : ['wiki']
    text  : # Welcome to Research Computing!

Welcome to Research Computing!

[User Services](https://rci.hpc.virginia.edu/wiki/User...

  - https://rci.hpc.virginia.edu/wiki/ACCORD%20Infrastructure
    title : Accord “Front Door”
    tags  : ['wiki']
    text  : # Accord “Front Door”

Accord “Front Door”

[https://accord.uvarc.io/](https://accord.uvarc.io/)

This site is a single-...

  - https://rci.hpc.virginia.edu/wiki/ACCORD
    title : ACCORD: Service & Support Plan
    tags  : ['wiki']
    text  : # ACCORD: Service & Support Plan

ACCORD: Service & Support Plan

1. Introduction

ACCORD is a collaboration across the...

  - https://rci.hpc.virginia.edu/wiki/Accounting
    title : Accounting
    tags  : ['wiki']
    text  : # Accounting

[HPC Accounting Workflow](https://rci.hpc.virginia.edu/wiki/HPC%20Accounting%20Workflow)...

  - https://rci.h

## 6. Generate Markdown Files

In [ ]:
from pathlib import Path
import re

# Locate this repo from either the project root or a notebook subfolder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app" / "kb_integration" / "tasks.py").is_file()
     and (path / "scrapers").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or a subfolder.")

# Store generated files inside this repo
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "wiki"

# Automatically create data/wiki if it doesn't exist
OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


def safe_filename(filename):

    filename = re.sub(
        r'[<>:"/\\|?*]',
        '_',
        filename
    )

    filename = re.sub(
        r'\s+',
        ' ',
        filename
    ).strip()

    return filename[:150]


print(
    f"Documents available: "
    f"{len(documents)}"
)

print(
    f"Writing files to: "
    f"{OUTPUT_FOLDER}"
)


created = 0
failed = 0


for url, doc in documents.items():

    try:

        title = (
            doc["metadata"].get("title")
            or page_title_from_url(url)
        )

        filename = f"wiki_{safe_filename(title)}.md"

        file_path = OUTPUT_FOLDER / filename

        with open(
            file_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                doc["text"].strip() + "\n"
            )

        created += 1

        print(
            f"Created: {file_path.name}"
        )

    except Exception as e:

        failed += 1

        print(
            f"ERROR processing "
            f"{url}: {e}"
        )


print("Markdown Generation Complete")

print(f"Created: {created} files")
print(f"Failed:  {failed} files")
print(f"Output folder: {OUTPUT_FOLDER}")